### Install the SDK

In [1]:
%pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 10.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.


### Connect to Gemini

In [2]:
from google import genai
from google.genai import types
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

MODEL = "gemini-2.5-flash"

print("Gemini connected!")

Gemini connected!


### Create the agent's tools

In [3]:
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    result = a * b
    print(f"Tool executed: {a} × {b} = {result}")
    return result


def add(a: float, b: float) -> float:
    """Add two numbers."""
    result = a + b
    print(f"Tool executed: {a} + {b} = {result}")
    return result


def subtract(a: float, b: float) -> float:
    """Subtract b from a."""
    result = a - b
    print(f"Tool executed: {a} - {b} = {result}")
    return result

### Build your first AI agent

In [6]:
tools = [multiply, add, subtract]

question = """
First multiply 15 by 8.
Then add 20 to the multiplication result.
Use the available tools to perform the calculations.
"""

response = client.models.generate_content(
    model=MODEL,
    contents=question,
    config=types.GenerateContentConfig(
        tools=tools
    )
)

print("USER GOAL:")
print(question)

print("\nAGENT ANSWER:")
print(response.text)

Tool executed: 15 × 8 = 120
Tool executed: 120 + 20 = 140
USER GOAL:

First multiply 15 by 8.
Then add 20 to the multiplication result.
Use the available tools to perform the calculations.


AGENT ANSWER:
The final result is 140.


### Give the agent a different goal

In [5]:
question = """
Multiply 12 by 7, then subtract 10.
Use the available tools and explain the final result.
"""

response = client.models.generate_content(
    model=MODEL,
    contents=question,
    config=types.GenerateContentConfig(
        tools=tools
    )
)

print(response.text)

Tool executed: 12 × 7 = 84
Tool executed: 84 - 10 = 74
After multiplying 12 by 7, we got 84. Then, subtracting 10 from 84 gives us a final result of 74.


###Build a Real-World AI Support Agent

###Create a mock order database

In [7]:
orders = {
    "ORD-1001": {
        "customer": "Ali",
        "status": "Shipped",
        "item": "Wireless Mouse",
        "amount": 25.00,
        "days_since_delivery": None
    },
    "ORD-1002": {
        "customer": "Sara",
        "status": "Delivered",
        "item": "Keyboard",
        "amount": 60.00,
        "days_since_delivery": 5
    },
    "ORD-1003": {
        "customer": "Ahmed",
        "status": "Delivered",
        "item": "Headphones",
        "amount": 80.00,
        "days_since_delivery": 20
    }
}

print("Sample order database ready!")

Sample order database ready!


###Create the agent's tools

In [8]:

def lookup_order(order_id: str) -> dict:
    """Look up an order by its ID and return its details."""

    order = orders.get(order_id.upper())

    if order is None:
        return {"error": "Order not found. Check the order ID."}

    return {"order_id": order_id.upper(), **order}


def check_refund_eligibility(order_id: str) -> dict:
    """
    Check whether an order qualifies for a refund.
    Policy: only delivered orders within 14 days of delivery qualify.
    This function checks eligibility but does not issue a refund.
    """

    order = orders.get(order_id.upper())

    if order is None:
        return {"error": "Order not found."}

    if order["status"] != "Delivered":
        return {
            "eligible": False,
            "reason": "The order has not been delivered yet."
        }

    days = order["days_since_delivery"]

    if days <= 14:
        return {
            "eligible": True,
            "refund_amount": order["amount"],
            "reason": "The delivered order is within the 14-day refund window."
        }

    return {
        "eligible": False,
        "reason": "The 14-day refund window has expired."
    }


###Let Gemini use the tools

In [9]:

support_tools = [
    lookup_order,
    check_refund_eligibility
]

support_instructions = """
You are a customer support AI agent for an online store.

Help customers with order status and refund eligibility.

Rules:
- Use lookup_order to check order details.
- Use check_refund_eligibility when asked about refunds.
- Use tool results as the source of truth.
- Never invent order details or refund decisions.
- If an order ID is missing, ask the customer for it.
- Explain that checking eligibility does not issue a refund.
- Be polite, clear, and concise.
"""

question = (
    "Where is my order ORD-1002, and can I get a refund?"
)

response = client.models.generate_content(
    model=MODEL,
    contents=question,
    config=types.GenerateContentConfig(
        system_instruction=support_instructions,
        tools=support_tools
    )
)

print("CUSTOMER:")
print(question)

print("\nAI SUPPORT AGENT:")
print(response.text)


CUSTOMER:
Where is my order ORD-1002, and can I get a refund?

AI SUPPORT AGENT:
Your order ORD-1002 has been delivered 5 days ago.

Good news! Your order ORD-1002 is eligible for a refund of $60 because it is within the 14-day refund window. Please note that checking eligibility does not automatically issue a refund.
